# 02 · Metodologia
Lê as 13 tabelas do 01 e produz fatores FF3-BR, sinal residual, sinal bruto e filtro de qualidade. Não escolhe ação: só gera números por ação e por data.


## 1. Parâmetros e carga dos dados
Parâmetros do documento e as 13 tabelas do 01, com conferência de que todas têm as mesmas datas e ações.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

JANELA = 36
FOLGA = 0
MESES_SINAL = 11
MINIMO_RESIDUOS = 8
HISTORICO_MINIMO = JANELA - FOLGA
HISTORICO_LEGADO = 30
DESVIO = 'janela'
MINIMO_ACOES = 30
MINIMO_CELULAS = 6
JANELA_PISO = 12
SPREAD_PISO = 0.05
ALIQUOTA_IR = 0.34
LIMITE_ALAVANCAGEM = 3.0

PASTA = Path('../data/processed')
NOMES = ['cadastro_universo', 'retorno_mensal', 'preco_sem_ajuste_mensal', 'valor_mercado_mensal',
         'liquidez_12m', 'contab_pl', 'contab_ebitda', 'contab_divida_liquida', 'contab_da',
         'elegivel', 'saidas', 'cdi_mensal', 'ibov_mensal']
dados = {n: pd.read_pickle(PASTA / f'{n}.pkl') for n in NOMES}

retorno = dados['retorno_mensal']
for n in NOMES:
    if n in ('cadastro_universo', 'saidas'):
        continue
    assert dados[n].index.equals(retorno.index), n
    if dados[n].ndim == 2:
        assert dados[n].columns.equals(retorno.columns), n
assert retorno.index.is_monotonic_increasing and retorno.index.is_month_end.all()
assert (dados['elegivel'].dtypes == bool).all()

print(retorno.index[0].date(), retorno.index[-1].date(), len(retorno), 'meses')
print('elegíveis por mês (média):', round(dados['elegivel'].sum(axis=1).mean(), 1))

## 2. Fatores FF3-BR
MKT, SMB e HML do mês s, formados com informação do fim de s−1: seis carteiras por tamanho e B/M, ponderadas por valor de mercado.

In [ ]:
def construir_fatores(retorno, valor_mercado, pl, elegivel, cdi):
    saida = pd.DataFrame(np.nan, index=retorno.index, columns=['MKT', 'SMB', 'HML'])
    datas = retorno.index
    for i in range(len(datas) - 1):
        t, s = datas[i], datas[i + 1]
        base = pd.DataFrame({'cap': valor_mercado.loc[t], 'pl': pl.loc[t], 'ret': retorno.loc[s]})
        base = base[elegivel.loc[t]].dropna(subset=['cap', 'ret'])
        if len(base) < MINIMO_ACOES:
            continue
        saida.at[s, 'MKT'] = np.average(base['ret'], weights=base['cap']) - cdi.loc[s]
        com_valor = base[base['pl'] > 0]
        if len(com_valor) < MINIMO_ACOES:
            continue
        bm = com_valor['pl'] / com_valor['cap']
        tamanho = np.where(com_valor['cap'] <= com_valor['cap'].median(), 'S', 'B')
        valor = np.where(bm <= bm.quantile(0.3), 'L', np.where(bm >= bm.quantile(0.7), 'H', 'M'))
        celulas = {g: np.average(x['ret'], weights=x['cap']) for g, x in com_valor.groupby([tamanho, valor])}
        if len(celulas) < MINIMO_CELULAS:
            continue
        saida.at[s, 'SMB'] = np.mean([celulas[('S', v)] for v in 'LMH']) - np.mean([celulas[('B', v)] for v in 'LMH'])
        saida.at[s, 'HML'] = np.mean([celulas[(k, 'H')] for k in 'SB']) - np.mean([celulas[(k, 'L')] for k in 'SB'])
    return saida

fatores_ff3 = construir_fatores(dados['retorno_mensal'], dados['valor_mercado_mensal'], dados['contab_pl'],
                                dados['elegivel'], dados['cdi_mensal'])

print(fatores_ff3.notna().sum())
print(fatores_ff3.dropna().index[0].date(), fatores_ff3.dropna().index[-1].date())
print(pd.DataFrame({'média a.a. (×12)': fatores_ff3.mean() * 12, 'vol a.a.': fatores_ff3.std() * np.sqrt(12)}).round(3))

## 3. Diagnóstico dos fatores
Meses sem fator, tamanho da base por mês e correlação entre os fatores e o excesso do Ibovespa (só descrição).

In [ ]:
cap_t = dados['valor_mercado_mensal'].shift(1)
base = dados['elegivel'].shift(1, fill_value=False) & cap_t.notna() & dados['retorno_mensal'].notna()
com_valor = base & (dados['contab_pl'].shift(1) > 0)
tamanho = pd.DataFrame({'base': base.sum(axis=1), 'pl_positivo': com_valor.sum(axis=1)})

print('meses sem fator:', fatores_ff3.index[fatores_ff3.isna().any(axis=1)].strftime('%Y-%m').tolist())
print(tamanho[fatores_ff3.notna().all(axis=1)].describe().loc[['min', 'mean', 'max']].round(1))

ibov_excesso = (dados['ibov_mensal'] - dados['cdi_mensal']).rename('IBOV')
print(pd.concat([fatores_ff3, ibov_excesso], axis=1).corr().round(2))

## 4. Sinal residual: regressão em janela de 36 meses
Em cada data t, regride o excesso de retorno da ação em MKT, SMB e HML nos 36 meses até t. O sinal é a soma dos 11 resíduos de t−11 a t−1, dividida pelo desvio-padrão dos resíduos da janela.

In [ ]:
def sinal_da_acao(y, X, minimo=None):
    minimo = HISTORICO_MINIMO if minimo is None else minimo
    valido = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    if np.isnan(X).any() or valido.sum() < minimo:
        return np.nan
    A = np.column_stack([np.ones(len(y)), X])
    coef = np.linalg.lstsq(A[valido], y[valido], rcond=None)[0]
    residuos = np.full(len(y), np.nan)
    residuos[valido] = y[valido] - A[valido] @ coef
    recentes = residuos[-1 - MESES_SINAL:-1]
    if np.isfinite(recentes).sum() < MINIMO_RESIDUOS:
        return np.nan
    base_desvio = residuos[valido] if DESVIO == 'janela' else recentes[np.isfinite(recentes)]
    return np.nansum(recentes) / np.std(base_desvio, ddof=1)

def sinal_na_data(t, retorno, cdi, fatores, elegiveis, minimo=None):
    pos = retorno.index.get_loc(t)
    meses = retorno.index[pos - JANELA + 1:pos + 1]
    X = fatores.loc[meses].to_numpy()
    excesso = retorno.loc[meses, elegiveis].sub(cdi.loc[meses], axis=0)
    return pd.Series({c: sinal_da_acao(excesso[c].to_numpy(), X, minimo) for c in excesso})

t = pd.Timestamp('2020-08-31')
elegiveis = dados['elegivel'].columns[dados['elegivel'].loc[t]]
s = sinal_na_data(t, dados['retorno_mensal'], dados['cdi_mensal'], fatores_ff3, elegiveis)
print(len(elegiveis), 'elegíveis;', s.notna().sum(), 'com sinal')
print(s.describe().round(2))
print(s.sort_values().dropna().tail(5).round(2))

## 5. Sinais para todas as datas
`sinal_residual` roda a regressão em cada data de decisão, de ago/2020 até nov/2025. `sinal_bruto` é o retorno acumulado de t−11 a t−1, na mesma janela e com o mesmo pulo, para que a única diferença entre os dois seja retirar os fatores.

In [ ]:
retorno = dados['retorno_mensal']
datas = retorno.index
primeira = datas.get_loc(fatores_ff3.dropna().index[0]) + JANELA - 1

sinal_residual = pd.DataFrame(np.nan, index=datas, columns=retorno.columns)
sinal_residual_legado = sinal_residual.copy()
for t in datas[primeira:-1]:
    elegiveis = retorno.columns[dados['elegivel'].loc[t]]
    sinal_residual.loc[t, elegiveis] = sinal_na_data(t, retorno, dados['cdi_mensal'], fatores_ff3, elegiveis)
    sinal_residual_legado.loc[t, elegiveis] = sinal_na_data(t, retorno, dados['cdi_mensal'], fatores_ff3, elegiveis, HISTORICO_LEGADO)

sinal_bruto = (1 + retorno).rolling(MESES_SINAL, min_periods=MINIMO_RESIDUOS).apply(np.nanprod, raw=True).shift(1) - 1
sinal_bruto = sinal_bruto.where(dados['elegivel'])

assert not (sinal_residual.notna() & ~dados['elegivel']).any().any()
assert not (sinal_residual_legado.notna() & ~dados['elegivel']).any().any()
assert not (sinal_bruto.notna() & ~dados['elegivel']).any().any()

print(datas[primeira].date(), 'a', datas[-2].date())
print(sinal_residual.notna().sum(axis=1).loc[datas[primeira]:].describe().round(1))
corr = sinal_residual.corrwith(sinal_bruto, axis=1).dropna()
print(len(corr), 'meses | correlação residual × bruto: média', round(corr.mean(), 2), 'mín', round(corr.min(), 2), 'máx', round(corr.max(), 2))

## 5.1 Conferência do histórico mínimo
O sinal de 36 de 36 tem que ser um subconjunto do de 30 de 36, com o mesmo valor nas ações em comum, e o de 30 de 36 tem que reproduzir o da rodada anterior.

In [ ]:
comuns = sinal_residual.notna()
assert (sinal_residual_legado.notna() | ~comuns).all().all()
dif = (sinal_residual - sinal_residual_legado).abs().to_numpy()[comuns.to_numpy()]
assert dif.max() < 1e-12
n36, n30 = sinal_residual.notna().sum(axis=1), sinal_residual_legado.notna().sum(axis=1)
print('sinais por data: 36 de 36 =', int(n36.loc['2020-08-31']), '| 30 de 36 =', int(n30.loc['2020-08-31']), 'em ago/2020')
print('ações a menos por data (36 de 36 contra 30 de 36): média', round((n30 - n36).loc[datas[primeira]:datas[-2]].mean(), 1), '| máximo', int((n30 - n36).max()))
print('diferença máxima do sinal nas ações em comum:', dif.max())

## 6. Filtro de qualidade
Aprova a ação elegível com ROIC acima do CDI composto de 12 meses mais 5 p.p., EBITDA positivo e dívida líquida/EBITDA até 3 (ou caixa líquido). Dado ausente reprova.

In [ ]:
ebit = dados['contab_ebitda'] - dados['contab_da'].where(dados['contab_da'] > 0)
capital_investido = (dados['contab_pl'] + dados['contab_divida_liquida']).where(lambda c: c > 0)
roic = ebit * (1 - ALIQUOTA_IR) / capital_investido
alavancagem = dados['contab_divida_liquida'] / dados['contab_ebitda'].where(dados['contab_ebitda'] > 0)
piso_roic = (1 + dados['cdi_mensal']).rolling(JANELA_PISO).apply(np.prod, raw=True) - 1

caixa_liquido_ou_baixo = (dados['contab_divida_liquida'] <= 0) | (alavancagem <= LIMITE_ALAVANCAGEM)
aprovada = (dados['elegivel']
            & roic.gt(piso_roic + SPREAD_PISO, axis=0)
            & dados['contab_ebitda'].gt(0)
            & caixa_liquido_ou_baixo)

el = dados['elegivel']
print('piso_roic: primeiro valor', piso_roic.first_valid_index().date(), '| último', round(piso_roic.iloc[-1], 3))
print('ROIC válido entre elegíveis:', round(roic.where(el).notna().sum().sum() / el.sum().sum(), 2))
print(aprovada.sum(axis=1).loc[datas[primeira]:datas[-2]].describe().round(1))

## 7. Relatório do filtro
Aprovadas por ano contra o tamanho do quintil (N), meses com 10 ou menos aprovadas e cobertura de balanço. Só contagem, nenhum retorno.

In [ ]:
datas_sinal = datas[primeira:-1]
com_sinal = sinal_residual.loc[datas_sinal].notna().sum(axis=1)
relatorio = pd.DataFrame({'com_sinal': com_sinal,
                          'N_quintil': np.ceil(0.2 * com_sinal),
                          'aprovadas': aprovada.loc[datas_sinal].sum(axis=1),
                          'elegiveis_sem_roic': (dados['elegivel'] & roic.isna()).loc[datas_sinal].sum(axis=1)})

print(relatorio.groupby(relatorio.index.year).mean().round(1))
print('meses com aprovadas <= 10:', (relatorio['aprovadas'] <= 10).sum())
inativo = relatorio['aprovadas'] <= relatorio['N_quintil']
print('meses com aprovadas <= N:', inativo.sum(), 'de', len(relatorio), '=', round(inativo.mean(), 2))

## 7.1 Composição setorial do filtro
Participação de cada setor entre as elegíveis e entre as aprovadas, nos meses de decisão, para ver se o filtro concentra a carteira em algum setor.

In [ ]:
cad = dados['cadastro_universo']
comp = pd.DataFrame({'elegiveis': dados['elegivel'].loc[datas_sinal].sum().groupby(cad['setor_economico']).sum(),
                     'aprovadas': aprovada.loc[datas_sinal].sum().groupby(cad['setor_economico']).sum()})
comp = comp / comp.sum() * 100
comp['dif'] = comp['aprovadas'] - comp['elegiveis']
print(comp.round(1).sort_values('elegiveis', ascending=False))

## 8. Testes
O sinal e os fatores não mudam se os dados depois da data de decisão forem trocados por lixo. Os resíduos somam zero, a regressão bate com uma conta independente e toda aprovada é elegível.

In [ ]:
t = pd.Timestamp('2023-06-30')
pos = datas.get_loc(t)
elegiveis = retorno.columns[dados['elegivel'].loc[t]]
completo = sinal_na_data(t, retorno, dados['cdi_mensal'], fatores_ff3, elegiveis)

ret_f, cdi_f, fat_f = retorno.copy(), dados['cdi_mensal'].copy(), fatores_ff3.copy()
for x in (ret_f, cdi_f, fat_f):
    x.iloc[pos + 1:] = 999
pd.testing.assert_series_equal(completo, sinal_na_data(t, ret_f, cdi_f, fat_f, elegiveis))

ate_t = {n: dados[n].iloc[:pos + 1] for n in ['retorno_mensal', 'valor_mercado_mensal', 'contab_pl', 'elegivel', 'cdi_mensal']}
fatores_ate_t = construir_fatores(ate_t['retorno_mensal'], ate_t['valor_mercado_mensal'], ate_t['contab_pl'],
                                  ate_t['elegivel'], ate_t['cdi_mensal'])
pd.testing.assert_frame_equal(fatores_ate_t, fatores_ff3.iloc[:pos + 1])

acao = completo.dropna().index[0]
meses = datas[pos - JANELA + 1:pos + 1]
y = (retorno.loc[meses, acao] - dados['cdi_mensal'].loc[meses]).to_numpy()
A = np.column_stack([np.ones(JANELA), fatores_ff3.loc[meses].to_numpy()])
v = ~np.isnan(y)
b = np.linalg.solve(A[v].T @ A[v], A[v].T @ y[v])
res = np.full(JANELA, np.nan)
res[v] = y[v] - A[v] @ b
assert abs(np.nansum(res)) < 1e-10
assert np.isclose(np.nansum(res[-1 - MESES_SINAL:-1]) / np.nanstd(res[v], ddof=1), completo[acao])

assert not (aprovada & ~dados['elegivel']).any().any()
print('testes ok')

## 9. Gravação
Grava os arquivos do 02 em `data/processed`, alinhados às mesmas datas e ações do 01, para o 03 e o 04 lerem.

In [ ]:
saidas_02 = {'fatores_ff3': fatores_ff3, 'sinal_residual': sinal_residual, 'sinal_residual_legado': sinal_residual_legado, 'sinal_bruto': sinal_bruto,
             'aprovada': aprovada, 'roic': roic, 'alavancagem': alavancagem}

for nome, tabela in saidas_02.items():
    assert tabela.index.equals(retorno.index), nome
    if nome != 'fatores_ff3':
        assert tabela.columns.equals(retorno.columns), nome
    pd.to_pickle(tabela, PASTA / f'{nome}.pkl')
    print(nome, tabela.shape)

assert (aprovada.dtypes == bool).all()

## 10. Comparação com o legado
Fatores e sinal residual do código antigo (`fatores_mensais` e `sinais_36`) contra os nossos, nas datas e ações em comum. Os fatores têm correlação de 0,997 a 1,000 e o sinal, de 0,999. A diferença não é zero porque universo, liquidez e dados do 01 mudaram. Os 146 sinais só no legado são todos de dez/2025.

In [ ]:
legado = [PASTA / 'entradas.pkl', PASTA / 'sinais_36.pkl']
if all(p.exists() for p in legado):
    _, fatores_leg = pd.read_pickle(legado[0])
    comuns = fatores_leg.index.intersection(fatores_ff3.dropna().index)
    print(len(comuns), 'meses de fator em comum')
    print(fatores_ff3.loc[comuns].corrwith(fatores_leg.loc[comuns]).round(3))

    leg = pd.read_pickle(legado[1])
    datas_c = leg.index.intersection(sinal_residual.index)
    cols_c = leg.columns.intersection(sinal_residual.columns)
    a = sinal_residual.loc[datas_c, cols_c]
    b = leg.loc[datas_c, cols_c].where(dados['elegivel'].loc[datas_c, cols_c])
    ambos = a.notna() & b.notna()
    dif = (a - b).where(ambos).abs().to_numpy()
    print(len(datas_c), 'datas;', len(cols_c), 'códigos em comum')
    print('sinal só no novo:', (a.notna() & b.isna()).sum().sum(), '| só no legado:', (b.notna() & a.isna()).sum().sum(), '| nos dois:', ambos.sum().sum())
    print('diferença absoluta: mediana', round(np.nanmedian(dif), 4), '| máx', round(np.nanmax(dif), 4))
    print('correlação média por data:', round(a.corrwith(b, axis=1).mean(), 3))
    print((b.notna() & a.isna()).sum(axis=1).loc[lambda x: x > 0])
else:
    print('arquivos do legado ausentes: comparação não executada')

## 11. Resumo 
**O que o 02 entrega** (arquivos em `data/processed`): `fatores_ff3`, `sinal_residual`, `sinal_residual_legado`, `sinal_bruto`, `aprovada`, `roic` e `alavancagem`, alinhados às 106 datas e 1.419 ações do 01.

**Resultados:**
- Fatores FF3-BR válidos em 100 meses (set/2017 a dez/2025). Correlação de 0,98 do MKT com o excesso do Ibovespa e de 0,04 entre SMB e HML.
- Sinal residual e sinal bruto em 64 datas de decisão (ago/2020 a nov/2025). Correlação média entre os dois: 0,61.
- Filtro de qualidade com piso de CDI composto de 12 meses mais 5 p.p. As contagens de aprovadas e de meses com aprovadas ≤ N estão na saída da seção 7.
- A diferença setorial entre elegíveis e aprovadas está na saída da seção 7.1.

**Conferências que passaram:** sinal e fatores não usam dados posteriores à data de decisão, os resíduos somam zero, a regressão bate com uma conta independente, toda aprovada é elegível e as tabelas contábeis respeitam a defasagem de 45 e 90 dias.

**Regras do sinal como parâmetros:** `FOLGA` (0 nesta rodada, histórico mínimo de 36 de 36), `MINIMO_RESIDUOS` (8, sem efeito com 36 de 36) e `DESVIO` (`'janela'`, desvio dos 36 meses). O sinal de 30 de 36 fica gravado como `sinal_residual_legado` só para reproduzir a série de referência no 03. O sinal de 36 de 36 é um subconjunto do de 30 de 36 com valores idênticos (diferença máxima 0).

**Comparação com o legado:** correlação de 0,997 a 1,000 nos fatores e de 0,999 no sinal. Os 146 sinais só presentes no legado são todos de dez/2025, que o 02 não calcula por não haver mês seguinte.

**Limitações:** betas com 36 observações são ruidosos; ações deslistadas ficam fora dos fatores; o valor de mercado é da empresa; o limite de alavancagem 3 é convenção de crédito; nos meses em que aprovadas ≤ N, a carteira compra todas as aprovadas e o sinal não seleciona: nesses meses o resultado mede o filtro, não o momentum.

